In [1]:
!pip install langchain langchain-huggingface huggingface-hub  duckduckgo-search

In [2]:
!pip install html2text faiss-cpu streamlit chromadb langchain-community

In [3]:
!pip install numexpr youtube_search wikipedia

In [4]:
import pandas as pd
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

def measure_model_performance(model_name, max_length, input_text,
                             model_size_billion_params):
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    # Tokenize the input text
    input_tokens = tokenizer(input_text, return_tensors="pt")

    # Measure the time taken to generate the output
    start_time = time.time()
    output = model.generate(**input_tokens, max_length=max_length)
    end_time = time.time()

    # Decode the generated tokens to text
    output_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Calculate the time taken for generation
    time_taken = end_time - start_time

    # Calculate the number of tokens generated
    num_tokens = output.size(1)

    # Calculate tokens per second
    tokens_per_second = num_tokens / time_taken

    # Calculate tokens per second per billion parameters
    tokens_per_second_per_billion_params = tokens_per_second / model_size_billion_params

    # Create the data list
    data = {
        "model_name": model_name.split('/')[1],
        "billion_parameters": model_size_billion_params,
        "tokens_per_second": tokens_per_second,
        "tokens_per_second_per_billion_parameters": tokens_per_second_per_billion_params,
        "answer": output_text
    }

    return data

# Example usage
model_name = "mistralai/Mistral-7B-Instruct-v0.1"
max_length = 512
input_text = "describe briefly what is an AI agent"
model_size_billion_params = 7

data = measure_model_performance(model_name, max_length,
                                 input_text, model_size_billion_params)

# Convert data to DataFrame and display
df = pd.DataFrame([data])
df


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame()

# List of models to evaluate
models = [
    "mistralai/Mistral-7B-Instruct-v0.1",
    "microsoft/Phi-3-mini-4k-instruct",
    "Qwen/Qwen2-7B-Instruct",
    "tri-ml/mamba-7b-rw",
    "meta-llama/Meta-Llama-3-8B-Instruct"
]

# List of input texts to use with each model
input_texts = [
    "[INST]describe briefly what is an AI agent[/INST]",
    "<|user|>describe briefly what is an AI agent<|end|><|assistant|>",
    "describe briefly what is an AI agent",
    "describe briefly what is an AI agent",
    "describe briefly what is an AI agent"
]

parameters = [7, 3, 7, 7, 8]

# Ensure both lists have the same length
assert len(models) == len(input_texts), "The number of models must match the number of input texts."

results = []
# Measure performance for each model and append to the DataFrame
for model_name, input_text, model_size_billion_params in zip(models, input_texts, parameters):
    data = measure_model_performance(model_name, max_length, input_text,
                                     model_size_billion_params)
    results.append(data)

df = pd.DataFrame(results)
df

In [ ]:
import matplotlib.pyplot as plt
# Plotting the bar plot for tokens per second
plt.figure(figsize=(10, 6))
plt.bar(df['model_name'], df['tokens_per_second'], color='skyblue')
plt.xlabel('Model Name')
plt.ylabel('Tokens per second')
plt.title('Tokens per second by model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('Token_second.jpg', format='jpeg')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(df['model_name'], df['tokens_per_second_per_billion_parameters'], color='skyblue')
plt.xlabel('Model Name')
plt.ylabel('Tokens per second \n for billion parameters')
plt.title('Tokens per second for billion parameters')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('Token_second_B_param.jpg', format='jpeg')
plt.show()

In [ ]:
# we can print the answer:
for i in range(5):
    print(df.loc[i,'model_name'])
    print(df.loc[i,'answer'])

In [ ]:
data = {
    "Model": [
        "Mistral-7B-Instruct-v0.1",
        "Phi-3-mini-4k-instruct",
        "Qwen2-7B-Instruct",
        "mamba-7b-rw",
        "Meta-Llama-3-8B-Instruct"
    ],
    "Overall Quality": [8, 9, 10, 1, 8],
    "Completeness": [8, 9, 10, 1, 8],
    "Truthfulness": [9, 9, 10, 1, 9]
}

df = pd.DataFrame(data)

# Plotting the data
df.set_index("Model").plot(kind="bar", figsize=(12, 6), color=['#1f77b4', '#ff7f0e', '#2ca02c'])
plt.title('Scores for AI models')
plt.xlabel('Models')
plt.ylabel('Scores')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Score Type')
plt.tight_layout()
plt.savefig('evaluation.jpg', format='jpeg')
plt.show()

In [ ]:
from langchain.tools import DuckDuckGoSearchRun

ddg_search = DuckDuckGoSearchRun()
ddg_search.run('Who is the current president of Italy?')

In [ ]:
from langchain.agents import Tool

tools = [
   Tool(
       name="DuckDuckGo Search",
       func=ddg_search.run,
       description="A web search tool to extract information from Internet.",
   )
]


In [ ]:
#!pip install google-serp-api
import os
SERPER_API_KEY = 'your_key'
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

from langchain.utilities import GoogleSerperAPIWrapper

google_search = GoogleSerperAPIWrapper()

tools.append(
   Tool(
       name="Google Web Search",
       func=google_search.run,
       description="Google search tool to extract information from Internet.",
   )
)

In [ ]:
from langchain.tools import WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

tools.append(
   Tool(
       name="Wikipedia Web Search",
       func=wikipedia.run,
       description="Useful tool to search Wikipedia.",
   )
)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.agents import load_tools
from langchain.agents import initialize_agent
model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
    #attn_implementation="flash_attention_2", # if you have an ampere GPU
)

# Define the text generation pipeline using HuggingFace transformers
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=500, top_k=50, temperature=0.1)

# Wrap the pipeline in a HuggingFacePipeline object
llm = HuggingFacePipeline(pipeline=pipe)


agent = initialize_agent(
   tools, llm, agent="zero-shot-react-description", verbose=True,
    handle_parsing_errors=True,
    max_iterations=1,
)

# Define the query
query = "Who is the current president of Italy? Who was the previous one?"

# Execute the agent query
response = agent.run(query)
print(response)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.agents import load_tools, AgentExecutor, initialize_agent
from langchain_core.prompts import PromptTemplate

# Load the model and tokenizer
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
)

# Define the text generation pipeline using HuggingFace transformers
pipe = pipeline("text-generation", model=model,
                tokenizer=tokenizer, max_new_tokens=500,
                top_k=50, temperature=0.1,
               do_sample=True)

# Wrap the pipeline in a HuggingFacePipeline object
llm = HuggingFacePipeline(pipeline=pipe)

# Load the necessary tools
tools = load_tools(["ddg-search",  "llm-math", "wikipedia"], llm=llm)

# Define the prompt template with explicit stop instructions
template = '''Answer the following question as best as you can. You have access to the following tools:

{tools}

Use the following format:

Question: {input}
Thought: You should think about what action to take
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Do not answer or ask any other questions. Stop once you have provided the Final Answer.


Begin!

Question: {input}
'''

# Create a PromptTemplate from the template
prompt = PromptTemplate.from_template(template)

# Initialize the agent using initialize_agent
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent="zero-shot-react-description",
    prompt=prompt,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=1,
    stop_sequence="Final Answer:"
)

# Define the query
query = "The biography of Napoleon"

# Execute the query
response = agent.run(query)
print(response)


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain.agents import load_tools
from langchain.agents import initialize_agent
model_id = "mistralai/Mistral-7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    load_in_4bit=True,
    #attn_implementation="flash_attention_2", # if you have an ampere GPU
)
